# LLM Inference Optimization Testbed (driver)

This notebook is a thin driver over the reusable `llm_benchmark` package. It runs
the full optimization pipeline — HuggingFace baseline, vLLM (as an OpenAI-compatible
`vllm serve` subprocess), AWQ, Flash Attention, Prefix Caching, Speculative Decoding,
and the Combined stack — and reports the results.

All logic lives in `llm_benchmark/`. Edit that package to change behavior; keep this
notebook for execution only.


In [ ]:
# Install vLLM first (GPU) - pulls a matching CUDA PyTorch + transformers
!pip install vllm

# Install remaining dependencies
!pip install -q accelerate requests openai aiohttp tqdm pandas matplotlib numpy nvidia-ml-py3 huggingface-hub


In [8]:
!git clone https://github.com/h-sami-ullah/llm-inference-benchmark.git

Cloning into 'llm-inference-benchmark'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 32 (delta 9), reused 28 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 20.00 KiB | 353.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.


In [9]:
import os
print("Current working directory:", os.getcwd()
      )


Current working directory: /content


In [10]:
!ls /content/llm-inference-benchmark

llm_benchmark	       notebooks       README.md
llm_benchmark_offline  pyproject.toml  requirements.txt


In [11]:
# Make the repo root importable (adjust if your notebook lives elsewhere)
import os
import sys

repo_root ="/content/llm-inference-benchmark"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Using repo root:", repo_root)


Using repo root: /content/llm-inference-benchmark


In [17]:
from llm_benchmark import (
    setup_environment,
    set_seed,
    run_full_pipeline,
    display_results,
    save_results_to_file,
)

setup_environment()
set_seed(42)

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Run the whole benchmark pipeline (HF baseline + all vLLM optimizations)
results = run_full_pipeline()


In [ ]:
save_results_to_file(results)
display_results(results)


In [ ]:
# Visualization
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

names = [r.name for r in results]
tps_values = [r.tps for r in results]
ttft_values = [r.ttft_ms if r.ttft_ms is not None else np.nan for r in results]
memory_values = [r.memory_gb for r in results]

axes[0].barh(names, tps_values, color="steelblue")
axes[0].axvline(x=60, color="red", linestyle="--", label="Target: 60 TPS")
axes[0].set_xlabel("Tokens Per Second")
axes[0].set_title("Aggregate Throughput (TPS)")
axes[0].legend()

axes[1].barh(names, ttft_values, color="coral")
axes[1].axvline(x=1000, color="red", linestyle="--", label="Target: <1000 ms")
axes[1].set_xlabel("Time to First Token (ms)")
axes[1].set_title("TTFT (missing values omitted)")
axes[1].legend()

axes[2].barh(names, memory_values, color="mediumseagreen")
axes[2].axvline(x=16, color="red", linestyle="--", label="Limit: 16 GB")
axes[2].set_xlabel("GPU Memory (GB)")
axes[2].set_title("GPU Memory Usage")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Final summary
from llm_benchmark import format_ttft

baseline = next((r for r in results if r.name == "HF Baseline"), None)
final = next((r for r in results if r.name == "Combined Stack"), None)

if baseline is None or final is None:
    print("Could not find both HF Baseline and Combined Stack results.")
else:
    print("\n" + "=" * 60)
    print("FINAL IMPROVEMENT SUMMARY")
    print("=" * 60)
    print("\nBaseline (Hugging Face) -> Combined Stack:")
    print(f"  TPS:    {baseline.tps:.1f} -> {final.tps:.1f} ({final.tps / baseline.tps:.2f}x)")
    print(f"  TPM:    {baseline.tpm:.0f} -> {final.tpm:.0f}")
    print(f"  TTFT:   {format_ttft(baseline.ttft_ms)} -> {format_ttft(final.ttft_ms)}")
    print(f"  Memory: {baseline.memory_gb:.2f} GB -> {final.memory_gb:.2f} GB")

    print("\nTargets achieved:")
    if final.ttft_ms is None:
        print("  TTFT < 1000 ms: NOT MEASURED")
    else:
        print(f"  TTFT < 1000 ms: {'YES' if final.ttft_ms < 1000 else 'NO'}")
    print(f"  TPS > 60:       {'YES' if final.tps > 60 else 'NO'}")
